# TPCH Business SQL Queries

In [0]:
Select
    Round(Sum(extended_price * (1-discount_rate)), 2) as net_revenue,
    count(distinct order_key) as total_orders,
    count(distinct customer_key) as active_customers,
    Round(Sum(extended_price * (1-discount_rate)) / count(Distinct order_key), 2) as average_order_value
From tpch_dev.gold.fact_sales

-- 

**Interpretation:** Total net revenue across all 7 years is ~$1.09T from 7.5M orders placed by 499,989 customers, yielding an average order value of ~$145K. The business has a large, active customer base with high-value transactions.

**Suggested management action:** Use these figures as baseline KPIs for all downstream analysis; track AOV monthly to detect shifts in customer spending behavior.

In [0]:
Select
    Year(order_date) as Year,
    Month(order_date) as Month,
    Round(Sum(extended_price * (1-discount_rate)), 2) as net_revenue
From tpch_dev.gold.fact_sales
Group by Year, Month
order by Year, Month

**Interpretation:** Monthly revenue is remarkably stable at ~$13.5–14.1B per month across all years, with February consistently dipping ~8% lower than other months. The data ends abruptly in August 1998, indicating a partial-year cutoff.

**Suggested management action:** Investigate the February dip to determine if it's seasonal or data-related; ensure reporting accounts for the partial 1998 data when comparing annual totals.

In [0]:
WITH annual_revenue AS (
    SELECT
        YEAR(order_date) AS order_year,
        ROUND(SUM(extended_price * (1 - discount_rate)), 2) AS net_revenue
    FROM tpch_dev.gold.fact_sales
    GROUP BY YEAR(order_date)
)
SELECT
    order_year,
    net_revenue,
    LAG(net_revenue) OVER (ORDER BY order_year)                        AS prev_year_revenue,
    ROUND(
        (net_revenue - LAG(net_revenue) OVER (ORDER BY order_year))
        / LAG(net_revenue) OVER (ORDER BY order_year) * 100,
        2
    )                                                              AS yoy_change_pct
FROM annual_revenue
ORDER BY order_year

**Interpretation:** Annual revenue is extremely stable at ~$165B from 1992–1997 (YoY changes within ±0.6%), confirming a mature, predictable business. The -41.32% drop in 1998 is a data artifact — only 8 months of data are present.

**Suggested management action:** Exclude 1998 from annual growth comparisons or annualize it; focus strategic planning on the stable 1992–1997 baseline.

In [0]:
SELECT
    c.customer_market_segment,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_customer AS c
    ON s.customer_key = c.customer_key
GROUP BY c.customer_market_segment
ORDER BY net_revenue DESC

**Interpretation:** All five market segments contribute nearly equally (~$217–219B each, within 1% of each other), indicating no single segment dominates the business. Furniture and Building are marginally top performers.

**Suggested management action:** Maintain diversified segment strategy; consider targeted growth campaigns in the slightly underperforming HOUSEHOLD segment.

In [0]:
SELECT
    s.customer_key,
    c.customer_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_customer AS c
    ON s.customer_key = c.customer_key
GROUP BY s.customer_key, c.customer_name
ORDER BY net_revenue DESC
LIMIT 10

**Interpretation:** The top 10 customers each generate $6.5–6.9M in net revenue, representing the highest-value individual accounts. The gap between #1 and #10 is only ~5.5%, showing a tight cluster of top spenders.

**Suggested management action:** Establish a key-account management program for these 10 customers; prioritize retention since losing any one would impact revenue meaningfully.

In [0]:
WITH customer_revenue AS (
    SELECT
        s.customer_key,
        c.customer_name,
        ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
    FROM tpch_dev.gold.fact_sales AS s
    JOIN tpch_dev.gold.dim_customer AS c
        ON s.customer_key = c.customer_key
    GROUP BY s.customer_key, c.customer_name
),
top_10 AS (
    SELECT *
    FROM customer_revenue
    ORDER BY net_revenue DESC
    LIMIT 10
)
SELECT
    customer_key,
    customer_name,
    net_revenue,
    ROUND(net_revenue / SUM(net_revenue) OVER () * 100, 2) AS pct_of_total_revenue
FROM top_10
ORDER BY net_revenue DESC

**Interpretation:** Each of the top 10 customers accounts for ~9.75–10.29% of the top-10 combined revenue, confirming a highly even distribution with no single customer dominating. There is no concentration risk among top accounts.

**Suggested management action:** No urgent action needed on concentration risk; focus on growing the next tier of customers to expand the revenue base.

In [0]:
SELECT
    s.product_key,
    p.product_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_product AS p
    ON s.product_key = p.product_key
GROUP BY s.product_key, p.product_name
ORDER BY net_revenue DESC
LIMIT 10

**Interpretation:** The top product generates $2.92M, with the top 10 ranging from $2.70M to $2.92M — a tight 8% spread. No single product is a runaway bestseller, suggesting a broad product portfolio with even demand.

**Suggested management action:** Ensure stock availability for these top performers; analyze whether they share common attributes that could inform product development.

In [0]:
SELECT
    s.product_key,
    p.product_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_product AS p
    ON s.product_key = p.product_key
GROUP BY s.product_key, p.product_name
ORDER BY net_revenue ASC
LIMIT 10

**Interpretation:** The bottom 10 products generate only $212K–240K each, roughly 13x lower than the top 10. These underperformers may be niche items, discontinued, or poorly marketed.

**Suggested management action:** Review bottom-performing products for potential discontinuation, repositioning, or bundling with higher-selling items to improve inventory turnover.

In [0]:
SELECT
    p.product_type,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_product AS p
    ON s.product_key = p.product_key
GROUP BY p.product_type
ORDER BY net_revenue DESC

**Interpretation:** Across 150 product types, revenue is remarkably even ($7.0B–$7.6B), with ECONOMY PLATED STEEL leading at $7.58B. No product type dominates, confirming a highly diversified portfolio.

**Suggested management action:** Maintain the broad portfolio; investigate whether ECONOMY-tier products can be upsold to higher-margin finishes to increase profitability.

In [0]:
SELECT
    s.supplier_key,
    sp.supplier_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_supplier AS sp
    ON s.supplier_key = sp.supplier_key
GROUP BY s.supplier_key, sp.supplier_name
ORDER BY net_revenue DESC
LIMIT 10

**Interpretation:** The top supplier (Supplier#000007491) generates $29.2M, with the top 10 tightly clustered between $28.5M and $29.2M. Supplier performance is very evenly distributed.

**Suggested management action:** Build strategic partnerships with these top 10 suppliers; negotiate volume rebates given their consistent and comparable performance levels.

In [0]:
WITH supplier_revenue AS (
    SELECT
        s.supplier_key,
        sp.supplier_name,
        ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
    FROM tpch_dev.gold.fact_sales AS s
    JOIN tpch_dev.gold.dim_supplier AS sp
        ON s.supplier_key = sp.supplier_key
    GROUP BY s.supplier_key, sp.supplier_name
),
top_10 AS (
    SELECT *
    FROM supplier_revenue
    ORDER BY net_revenue DESC
    LIMIT 10
)
SELECT
    supplier_key,
    supplier_name,
    net_revenue,
    ROUND(net_revenue / SUM(net_revenue) OVER () * 100, 2) AS pct_of_top10_revenue
FROM top_10
ORDER BY net_revenue DESC


**Interpretation:** Each top-10 supplier holds ~9.85–10.12% of the group's combined revenue, confirming no supplier dependency risk. The supply base is well-diversified.

**Suggested management action:** No single-supplier risk exists; focus on maintaining quality and delivery SLAs across all top suppliers rather than consolidating.

In [0]:
SELECT
    g.region_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_customer AS c
    ON s.customer_key = c.customer_key
JOIN tpch_dev.gold.dim_geography AS g
    ON c.nation_key = g.geography_key
GROUP BY g.region_name
ORDER BY net_revenue DESC

**Interpretation:** Europe leads with $219.0B, followed closely by Middle East ($218.8B), Asia ($218.4B), America ($217.0B), and Africa ($216.7B). The 5 regions are within 1% of each other — revenue is globally distributed.

**Suggested management action:** Maintain balanced regional investment; consider expanding in Africa (lowest region) where there may be untapped growth potential.

In [0]:
SELECT
    g.nation_name,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_customer AS c
    ON s.customer_key = c.customer_key
JOIN tpch_dev.gold.dim_geography AS g
    ON c.nation_key = g.geography_key
GROUP BY g.nation_name
ORDER BY net_revenue DESC

**Interpretation:** All 25 nations contribute $43.0–44.2B each — a remarkably even spread with Iraq leading and Morocco at the bottom. No single nation dominates the revenue base.

**Suggested management action:** No country-specific concentration risk; use nation-level data to identify emerging markets for targeted expansion campaigns.

In [0]:
SELECT
    s.return_flag,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue,
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT s.order_key) AS total_orders
FROM tpch_dev.gold.fact_sales AS s
GROUP BY s.return_flag
ORDER BY net_revenue DESC

**Interpretation:** About 50.6% of line items (flag N) were never returned, while 24.7% were returned (R) and 24.7% accepted (A). Returned items represent $269.1B in revenue — roughly a quarter of total volume.

**Suggested management action:** Investigate the high return rate (~25%); implement quality controls and return-prevention initiatives to recover revenue from the R-flagged segment.

In [0]:
SELECT
    c.customer_market_segment,
    ROUND(AVG(s.discount_rate) * 100, 2) AS avg_discount_pct
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_customer AS c
    ON s.customer_key = c.customer_key
GROUP BY c.customer_market_segment
ORDER BY avg_discount_pct DESC

**Interpretation:** All five market segments receive an identical 5.00% average discount, indicating a uniform pricing policy with no segment-specific discounting strategy.

**Suggested management action:** Evaluate whether a differentiated discounting strategy could boost revenue in underperforming segments or reward high-volume segments with deeper discounts.

In [0]:
SELECT
    ship_mode,
    ROUND(AVG(DATEDIFF(receipt_date, ship_date)), 2) AS avg_ship_to_receipt_days
FROM tpch_dev.gold.fact_sales
GROUP BY ship_mode
ORDER BY avg_ship_to_receipt_days DESC

**Interpretation:** All shipping modes show nearly identical transit times (~15.5 days), with FOB marginally higher at 15.51 days. The choice of shipping mode does not meaningfully affect delivery speed.

**Suggested management action:** Optimize shipping cost rather than speed — since all modes perform similarly, consolidate shipments via the most cost-effective mode.

In [0]:
SELECT
    order_ship_priority,
    COUNT(DISTINCT order_key) AS total_orders,
    ROUND(SUM(extended_price * (1 - discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales
GROUP BY order_ship_priority
ORDER BY net_revenue DESC

**Interpretation:** All 7.5M orders have a ship priority of 0, meaning no prioritization is applied across the order base. This suggests the priority field is either deprecated or unused in the current system.

**Suggested management action:** Investigate whether order priority should be populated and used to differentiate fulfillment SLAs for high-value or urgent orders.

In [0]:
SELECT
    YEAR(order_date) AS order_year,
    MONTH(order_date) AS order_month,
    order_status,
    COUNT(DISTINCT order_key) AS total_orders,
    ROUND(SUM(extended_price * (1 - discount_rate)), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales
GROUP By order_year, order_month, order_status
ORDER BY order_year, order_month, total_orders DESC

**Interpretation:** Orders transition from fully Fulfilled (F) in 1992–1994 to a mixed status in mid-1995, then to fully Open (O) from July 1995 onward. This shift suggests a change in order processing lifecycle rather than deteriorating fulfillment performance.

**Suggested management action:** Investigate the 1995 transition period to understand the status change; ensure that "Open" orders from later years are not stuck in the fulfillment pipeline.

In [0]:
WITH product_segment_revenue AS (
    SELECT
        c.customer_market_segment,
        s.product_key,
        p.product_name,
        ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue
    FROM tpch_dev.gold.fact_sales AS s
    JOIN tpch_dev.gold.dim_product AS p
        ON s.product_key = p.product_key
    JOIN tpch_dev.gold.dim_customer AS c
        ON s.customer_key = c.customer_key
    GROUP BY c.customer_market_segment, s.product_key, p.product_name
)
SELECT
    customer_market_segment,
    product_key,
    product_name,
    net_revenue,
    RANK() OVER (
        PARTITION BY customer_market_segment
        ORDER BY net_revenue DESC
    ) AS revenue_rank
FROM product_segment_revenue
ORDER BY customer_market_segment, revenue_rank

**Interpretation:** Each market segment ranks hundreds of products by revenue, with rank 1 in each segment being the top performer for that segment's customers. This enables segment-specific product strategy and cross-selling insights.

**Suggested management action:** Use these rankings to tailor product promotions per segment — promote rank 1 products as segment champions and investigate low-ranked products for segment-specific issues.

In [0]:
SELECT
    cg.region_name AS customer_region,
    sg.region_name AS supplier_region,
    ROUND(SUM(s.extended_price * (1 - s.discount_rate)), 2) AS net_revenue,
    COUNT(DISTINCT s.order_key) AS total_orders
FROM tpch_dev.gold.fact_sales AS s
JOIN tpch_dev.gold.dim_customer AS c
    ON s.customer_key = c.customer_key
JOIN tpch_dev.gold.dim_geography AS cg
    ON c.nation_key = cg.geography_key
JOIN tpch_dev.gold.dim_supplier AS sp
    ON s.supplier_key = sp.supplier_key
JOIN tpch_dev.gold.dim_geography AS sg
    ON sp.supplier_nation_key = sg.geography_key
GROUP BY cg.region_name, sg.region_name
ORDER BY customer_region, net_revenue DESC

**Interpretation:** The 25 customer-region × supplier-region combinations show very even revenue (~$43–44B each), with cross-regional supply chains being as common as intra-regional ones. No single supply route dominates the business.

**Suggested management action:** Assess whether nearshoring (intra-regional supply) could reduce logistics costs without significantly impacting revenue, given the even distribution across routes.